In [27]:
import pandas as pd
data = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [28]:
X = data.drop('Outcome',axis=1)
y = data['Outcome']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
y_scores = model.predict_proba(X_test)[:,1]
print(y_scores)

[0.0495376  0.17459071 0.09362513 0.2551022  0.63549763 0.11671255
 0.06567149 0.42191788 0.04865943 0.57564545 0.33874537 0.41301962
 0.69855052 0.19964051 0.02001651 0.82469077 0.86657545 0.03101256
 0.25510051 0.89492636 0.95245335 0.83477975 0.11740834 0.44670755
 0.08923396 0.0688207  0.65116518 0.41197506 0.17861729 0.28689366
 0.24548703 0.43901236 0.00942506 0.24259232 0.35169128 0.96164473
 0.34359585 0.80923856 0.29380795 0.05083306 0.1818489  0.08672337
 0.42228328 0.18738343 0.03011727 0.0394937  0.24741239 0.42782187
 0.10529443 0.37970895 0.99355865 0.10667889 0.39101596 0.7688143
 0.36866148 0.45774882 0.95035341 0.41991928 0.39156094 0.04376209
 0.37293033 0.901279   0.88176163 0.88842918 0.33084351 0.0693201
 0.95264199 0.20055758 0.26055491 0.35462971 0.10359055 0.08168641
 0.45044035 0.09221339 0.0864889  0.50846151 0.36842004 0.07540211
 0.21421161 0.22594787 0.05049586 0.21043887 0.07232146 0.22887558
 0.67422076 0.31019998 0.09028672 0.1100514  0.2052806  0.569758

In [29]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
print(thresholds)

[       inf 0.99355865 0.95264199 0.95245335 0.82469077 0.7688143
 0.76201063 0.68388593 0.67532491 0.67422076 0.65116518 0.65084109
 0.63549763 0.58629146 0.57564545 0.56975889 0.52518837 0.47019234
 0.46128359 0.45044035 0.43901236 0.43864837 0.42228328 0.41991928
 0.41301962 0.41197506 0.40477544 0.37970895 0.36866148 0.31019998
 0.30880321 0.27772077 0.27203801 0.25845709 0.2551022  0.24741239
 0.24548703 0.23663601 0.22887558 0.1818489  0.17861729 0.17459071
 0.17327153 0.14637982 0.14076711 0.11671255 0.11494414 0.11000453
 0.10667889 0.08923396 0.08672337 0.02001651 0.01931265 0.00160784]


In [30]:
import plotly.graph_objects as go 
import numpy as np

# generate a trace for ROC curve
trace0 = go.Scatter(
    x = fpr,
    y = tpr,
    mode = 'lines',
    name='ROC curve'
)

# only label every nth point to avoid cluttering
n=10
indices = np.arange(len(thresholds)) % n == 0 # choose indices where index mod n is 0

trace1 = go.Scatter(
    x = fpr[indices],
    y = tpr[indices],
    mode = 'markers+text',
    name='Therehold points',
    text=[f"Thr={thr:.2f}" for thr in thresholds[indices]],
    textposition='top center'
)

# Digonal line
trace2 = go.Scatter(
    x = [0,1],
    y = [0,1],
    mode = 'lines',
    name='Randomk (Area = 0.5)',
    line = dict(dash='dash')
)

data = [trace0,trace1,trace2]

# Define layout with squre aspect ratio

layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=False   
)

# Define figure and add data
fig = go.Figure(data = data, layout=layout)
fig.show()


In [31]:
# Assume that fpr tpr thresholds have already been calculated
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print("Optimal threshold is: ",optimal_threshold)

Optimal threshold is:  0.3686614815032027


In [32]:
from sklearn.metrics import roc_auc_score

# Assuming fpr, tpr, thresholds are already calculated as before
fpr, tpr, thresholds = roc_curve(y_test, y_scores)

# Calculate the AUC (Area Under the Curve)
roc_auc = roc_auc_score(y_test, y_scores)

# generate a trace for ROC curve
trace0 = go.Scatter(
    x = fpr,
    y = tpr,
    mode = 'lines',
    name=f'ROC curve(Area = {roc_auc:.2f})'
)

# only label every nth point to avoid cluttering
n=10
indices = np.arange(len(thresholds)) % n == 0 # choose indices where index mod n is 0

trace1 = go.Scatter(
    x = fpr[indices],
    y = tpr[indices],
    mode = 'markers+text',
    name='Therehold points',
    text=[f"Thr={thr:.2f}" for thr in thresholds[indices]],
    textposition='top center'
)

# Digonal line
trace2 = go.Scatter(
    x = [0,1],
    y = [0,1],
    mode = 'lines',
    name='Randomk (Area = 0.5)',
    line = dict(dash='dash')
)

data = [trace0,trace1,trace2]

# Define layout with squre aspect ratio

layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True  
)

# Define figure and add data
fig = go.Figure(data = data, layout=layout)
fig.show()


In [33]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train,y_train)
lr_scores = lr_model.predict_proba(X_test)[:,1]

svm_model = SVC(probability=True)
svm_model.fit(X_train,y_train)
svm_scores = svm_model.predict_proba(X_test)[:,1]

# Assuming fpr, tpr, thresholds are already calculated as before
lr_fpr, lr_tpr, lr_thresholds = roc_curve(y_test, y_scores)
lr_auc = roc_auc_score(y_test,lr_scores)
svm_fpr, svm_tpr,svm_thresholds = roc_curve(y_test, svm_scores)
svm_auc = roc_auc_score(y_test,svm_scores)

# generate a trace for ROC curve
trace0 = go.Scatter(
    x = fpr,
    y = tpr,
    mode = 'lines',
    name=f'ROC curve(Area = {lr_auc:.2f})'
)

# only label every nth point to avoid cluttering
trace1 = go.Scatter(
    x = svm_fpr,
    y = svm_tpr,
    mode = 'lines',
    name=f'SVM (Area = {lr_auc:.2f})'
)

# Digonal line
trace2 = go.Scatter(
    x = [0,1],
    y = [0,1],
    mode = 'lines',
    name='Randomk (Area = 0.5)',
    line = dict(dash='dash')
)

data = [trace0,trace1,trace2]

# Define layout with squre aspect ratio

layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True  
)

# Define figure and add data
fig = go.Figure(data = data, layout=layout)
fig.show()


c:\AI_ML\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
